## Enviroment Setup & Project Paths
Setup the base project path and make sure we can import modules from the src folder. 

In [ ]:
import sys, os
from pathlib import Path

# --- Detect whether we're in a notebook ---
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    # Notebook context: use current working directory instead
    ROOT = Path(os.getcwd()).resolve()

# --- Add src folder to the path if needed ---
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

print("ROOT set to:", ROOT)


## load environment variables and initialize the ADE client
This step loads the project’s environment variables (stored in .env) and initializes the LandingAI ADE client using the API key.

In [ ]:
from dotenv import load_dotenv
import os
from landingai_ade import LandingAIADE

# Load environment variables from .env
load_dotenv(ROOT / ".env")

# Retrieve API key from environment
api_key = os.getenv("VISION_AGENT_API_KEY")

if not api_key:
    raise ValueError("API key not found. Make sure it's defined in your .env file as VISION_AGENT_API_KEY")

# Initialize ADE client
ade_client = LandingAIADE(apikey=api_key)

print("ADE client initialized successfully!")



✅ ADE client initialized successfully!


##  Validate working directories and Input Verification
This cell discovers PDFs in subfolders of input_folder/, infers the city from the folder name, and prints a quick inventory.

In [ ]:
# City-aware input discovery

from pathlib import Path

INPUT_DIR = ROOT / "input_folder"
RESULTS_DIR = ROOT / "results_folder"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def discover_city_pdfs(input_root: Path) -> list[tuple[str, Path]]:
    """
    Return a list of (city_name, pdf_path) for all PDFs in immediate subfolders of input_root.
    City name is derived from the folder name (title-cased, underscores/dashes replaced with spaces).
    """
    items: list[tuple[str, Path]] = []
    if not input_root.exists():
        return items

    for sub in sorted(p for p in input_root.iterdir() if p.is_dir()):
        city = sub.name.replace("_", " ").replace("-", " ").strip().title()
        pdfs = sorted(sub.glob("*.pdf"))
        for p in pdfs:
            items.append((city, p))
    return items

city_pdfs = discover_city_pdfs(INPUT_DIR)

# Inventory
from collections import Counter
counts = Counter([c for c, _ in city_pdfs])
print(f"Found {len(city_pdfs)} PDFs across {len(counts)} cities in {INPUT_DIR}")
for city, n in counts.most_common():
    print(f" - {city}: {n} PDFs")

# Preview a few
for city, p in city_pdfs[:5]:
    print(f"   {city} :: {p.name}")


## Extraction (ADE SDK, table-first), resumable, saves CSV + Parquet

## import and normalization storage (A)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# ROOT detection (as you already have)
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    ROOT = Path(os.getcwd()).resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from config import INPUT_DIR, RESULTS_DIR
from normalizers import normalize_any
from db import write_cases
import pandas as pd


In [ ]:
# Extraction via ADE SDK (table chunks -> DataFrame), resumable

import json
import pandas as pd
from io import StringIO

csv_path     = RESULTS_DIR / "ade_extracted_results.csv"
parquet_path = RESULTS_DIR / "ade_extracted_results.parquet"

def markdown_table_to_df(md: str) -> pd.DataFrame:
    """
    Convert a GitHub-style pipe markdown table into a DataFrame.
    Assumes header line and separator line are present.
    """
    lines = [ln.strip() for ln in md.strip().splitlines() if ln.strip()]
    if len(lines) < 2:
        return pd.DataFrame()
    # Remove leading/trailing pipes and split
    rows = [ [cell.strip() for cell in ln.strip("|").split("|")] for ln in lines ]
    header, sep, *data = rows
    # Some markdown tables include alignment markers like :---:
    # Filter out the separator line if it looks like dashes
    if all(set(c) <= set("-: ") for c in sep):
        df = pd.DataFrame(data, columns=header)
    else:
        # Fallback: treat all lines as data with first row as header
        df = pd.DataFrame(rows[1:], columns=rows[0])
    return df

def parsed_tables_to_df(parsed: dict) -> pd.DataFrame:
    """
    From ADE parse response, collect all chunks of type 'table' and stack as one DataFrame.
    Each table may have different columns; we align by outer join, then tidy later.
    """
    chunks = (parsed or {}).get("chunks") or []
    frames = []
    for ch in chunks:
        if (ch or {}).get("type") == "table":
            md = ch.get("markdown") or ""
            if md.strip():
                try:
                    df = markdown_table_to_df(md)
                    if not df.empty:
                        frames.append(df)
                except Exception:
                    # If any table fails to parse, skip it; we'll retain others
                    pass
    if not frames:
        return pd.DataFrame()
    # Outer join to keep all columns from all tables
    out = pd.concat(frames, axis=0, ignore_index=True, sort=False)
    # Normalize header spacing once
    out.columns = [c.strip().replace("\n", " ").replace("  ", " ") for c in out.columns]
    return out

# Load existing to resume
if csv_path.exists():
    existing = pd.read_csv(csv_path)
    # Track processed pairs as (city, source_file)
    processed = set(zip(existing.get("city", []), existing.get("source_file", [])))
    frames = [existing]
    print(f"Resuming from existing CSV with {len(existing)} rows across "
          f"{existing['source_file'].nunique() if 'source_file' in existing else 0} files.")
else:
    processed = set()
    frames = []
    print("No previous extraction found. Starting fresh.")

for city, pdf in city_pdfs:
    key = (city, pdf.name)
    if key in processed:
        continue
    try:
        parsed = ade_client.parse(document=pdf, model="dpt-2")
        df_tables = parsed_tables_to_df(parsed)
        if df_tables.empty:
            print(f"Skipped (no table rows): {city} :: {pdf.name}")
            continue

        # Attach metadata
        df_tables["city"] = city
        df_tables["source_file"] = pdf.name

        frames.append(df_tables)

        # Incremental save after each PDF
        all_df = pd.concat(frames, ignore_index=True, sort=False)
        all_df.to_csv(csv_path, index=False)
        all_df.to_parquet(parquet_path, index=False)
        print(f"Processed {city} :: {pdf.name} (+{len(df_tables)} rows, total {len(all_df)})")

    except Exception as e:
        print(f"Error on {city} :: {pdf.name} -> {e}")

print("Extraction complete.")
print("Outputs:")
print(" -", csv_path)
print(" -", parquet_path)
